In [ ]:
import pandas as pd

df = pd.read_csv("../data/SampleSuperstore.csv",  encoding="latin1")

df.head()

In [ ]:
# Check for data quality issues
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())

#Validate Data types and Business rules
Pay special attention to:
- Order Date → should be datetime
- Ship Date → should be datetime
- Sales → numeric
- Profit → numeric
- Quantity → numeric
- Discount → numeric

In [ ]:
df[["Sales", "Quantity", "Discount", "Profit"]].describe()

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

print("Invalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())

print("Orders shipped before ordering:",
      (df["Ship Date"] < df["Order Date"]).sum())

In [ ]:

# Check how many records have negative profit
loss_records = (df["Profit"] < 0).sum()

print("Records with a loss:", loss_records)
print("Percentage of records with a loss:",
      round(loss_records / len(df) * 100, 2), "%")


# Analyze profit by discount level
discount_analysis = df.groupby("Discount").agg(
    Records=("Profit", "count"),
    Average_Sales=("Sales", "mean"),
    Average_Profit=("Profit", "mean"),
    Total_Profit=("Profit", "sum"),
    Loss_Records=("Profit", lambda x: (x < 0).sum())
).round(2)

discount_analysis


In [ ]:

category_analysis = df.groupby("Category").agg(
    Records=("Profit", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean"),
    Loss_Records=("Profit", lambda x: (x < 0).sum())
).round(2)

category_analysis


In [ ]:
subcategory_analysis = df.groupby("Sub-Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean"),
    Records=("Profit", "count")
).sort_values("Total_Profit", ascending=False).round(2)

subcategory_analysis

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [ ]:
df.info()

In [ ]:

df["Year_Month"] = df["Order Date"].dt.to_period("M")

monthly_analysis = df.groupby("Year_Month").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).round(2)

monthly_analysis


In [ ]:

import matplotlib.pyplot as plt

monthly_analysis["Total_Sales"].plot(
    figsize=(12, 5),
    marker="o",
    title="Monthly Sales Trend"
)

plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_analysis.index.astype(str),
    monthly_analysis["Total_Sales"],
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print(monthly_analysis)

In [ ]:
# Annual business performance
annual_analysis = df.groupby(
    df["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Sales", "count")
)

# Calculate annual sales growth
annual_analysis["Sales_Growth_%"] = (
    annual_analysis["Total_Sales"].pct_change() * 100
)

# Calculate profit margin
annual_analysis["Profit_Margin_%"] = (
    annual_analysis["Total_Profit"] /
    annual_analysis["Total_Sales"] * 100
)

print(annual_analysis.round(2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual Sales
annual_analysis["Total_Sales"].plot(
    kind="bar",
    ax=axes[0],
    title="Annual Sales Performance"
)

axes[0].set_ylabel("Sales ($)")
axes[0].set_xlabel("Year")
axes[0].tick_params(axis="x", rotation=0)

# Annual Profit
annual_analysis["Total_Profit"].plot(
    kind="bar",
    ax=axes[1],
    title="Annual Profit Performance"
)

axes[1].set_ylabel("Profit ($)")
axes[1].set_xlabel("Year")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Profit Margin Investigation

df.columns.tolist()

In [ ]:
discount_analysis = df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)
discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"] /
    discount_analysis["Total_Sales"] * 100
)
discount_analysis.round(2)

In [ ]:
#Which product category generates the most profit?
category_analysis = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)
category_analysis["Profit_Margin_%"] = (
    category_analysis["Total_Profit"] /
    category_analysis["Total_Sales"] * 100
)
category_analysis.sort_values("Total_Profit", ascending=False).round(2)

In [ ]:
#Investigate Furniture: Now we move from category analysis to Sub-Category Analysis.
subcategory_analysis = df.groupby("Sub-Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

subcategory_analysis["Profit_Margin_%"] = (
    subcategory_analysis["Total_Profit"] /  
    subcategory_analysis["Total_Sales"] * 100
)
subcategory_analysis.sort_values("Profit_Margin_%", ascending=False).round(2)  




In [ ]:
#"Which specific products are causing us to lose money?"
product_analysis = df.groupby(["Product ID", "Product Name", "Category", "Sub-Category"]).agg(

    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
)   

product_analysis["Profit_Margin_%"] = (
    product_analysis["Total_Profit"] /
    product_analysis["Total_Sales"] * 100
)

loss_products = product_analysis[
    product_analysis["Total_Profit"] < 0
].sort_values("Total_Profit")

loss_products.head(10).round(2)

In [ ]:
#Are discounts causing product losses?
top_loss_products = [
    "TEC-MA-10000418",
    "TEC-MA-10000822",
    "TEC-MA-10004125"
]

loss_transactions = df[
    df["Product ID"].isin(top_loss_products)
]

discount_investigation = loss_transactions.groupby(
    ["Product Name", "Discount"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
)

discount_investigation.round(2)

In [ ]:
#Calculate profit margins for each discount level
discount_investigation = discount_investigation.copy()

discount_investigation["Profit_Margin_%"] = (
    discount_investigation["Total_Profit"] /
    discount_investigation["Total_Sales"] * 100
)

discount_investigation.round(2)

In [ ]:
#Let's create your first professional chart
import matplotlib.pyplot as plt

# Prepare chart data
chart_df = discount_investigation.reset_index()

chart_df["Discount_Label"] = (
    chart_df["Discount"].mul(100).astype(int).astype(str) + "%"
)

chart_df["Product_Label"] = (
    chart_df["Product Name"].str.replace(
        "Cubify CubeX 3D Printer ", "", regex=False
    ).str.replace(
        "Lexmark MX611dhe Monochrome Laser Printer",
        "Lexmark MX611dhe",
        regex=False
    )
)

chart_df["Label"] = (
    chart_df["Product_Label"] +
    " - " +
    chart_df["Discount_Label"]
)

# Create chart
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    chart_df["Label"],
    chart_df["Profit_Margin_%"]
)

ax.axvline(0, linewidth=1)

ax.set_title("Discount Impact on Product Profitability")
ax.set_xlabel("Profit Margin (%)")
ax.set_ylabel("Product and Discount Level")

ax.invert_yaxis()

# Add percentage labels
for bar in bars:
    width = bar.get_width()
    ax.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f" {width:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
segment_analysis = df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

segment_analysis["Profit_Margin_%"] = (
    segment_analysis["Total_Profit"] /
    segment_analysis["Total_Sales"] * 100
)

segment_analysis.sort_values(
    "Total_Profit",
    ascending=False
).round(2)

In [ ]:
#Regional Sales Analysis — next
region_analysis = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

region_analysis["Profit_Margin_%"] = (
    region_analysis["Total_Profit"] /
    region_analysis["Total_Sales"] * 100
)

region_analysis.sort_values(
    "Total_Profit",
    ascending=False
).round(2)

In [ ]:
#Monthly Sales Trends
# Convert Order Date to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])

# Analyze monthly performance
monthly_analysis = df.groupby(
    df["Order Date"].dt.to_period("M")
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

# Calculate profit margin
monthly_analysis["Profit_Margin_%"] = (
    monthly_analysis["Total_Profit"] /
    monthly_analysis["Total_Sales"] * 100
)

# Calculate month-over-month sales growth
monthly_analysis["Sales_Growth_%"] = (
    monthly_analysis["Total_Sales"].pct_change() * 100
)

monthly_analysis.round(2)

In [ ]:
#We will create a Monthly Sales Trend Chart using Matplotlib.
import matplotlib.pyplot as plt

# Create monthly sales chart
plt.figure(figsize=(15, 6))

plt.plot(
    monthly_analysis.index.astype(str),
    monthly_analysis["Total_Sales"],
    marker="o"
)

plt.title("Monthly Sales Trend (2023–2026)")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=90)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Shipping Mode Analysis
# Calculate delivery time in days
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df["Order Date"] = pd.to_datetime(df["Order Date"])
# Calculate delivery time in days
df["Delivery_Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days 

# Analyze shipping modes
shipping_analysis = df.groupby("Ship Mode").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Average_Delivery_Days=("Delivery_Days", "mean")
)

# Calculate profit margin
shipping_analysis["Profit_Margin_%"] = (
    shipping_analysis["Total_Profit"] /
    shipping_analysis["Total_Sales"] * 100
)

shipping_analysis.sort_values(
    "Total_Sales", ascending=False
).round(2)

In [ ]:
#Customer Analysis
customer_analysis = df.groupby(
    ["Customer ID", "Customer Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Total_Quantity=("Quantity", "sum")
)

customer_analysis["Profit_Margin_%"] = (
    customer_analysis["Total_Profit"] /
    customer_analysis["Total_Sales"] * 100
)

# Top 10 customers by total sales
top_customers = customer_analysis.sort_values(
    "Total_Sales",
    ascending=False
).head(10)

top_customers.round(2)

In [ ]:
#Find the most and least profitable customers
# Top 10 customers by profit
top_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=False
).head(10)

print("TOP 10 MOST PROFITABLE CUSTOMERS")
display(top_profit_customers.round(2))


# Bottom 10 customers by profit
bottom_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=True
).head(10)

print("TOP 10 LEAST PROFITABLE CUSTOMERS")
display(bottom_profit_customers.round(2))

The customer profitability analysis identified several customers generating negative profits. Cindy Stewart recorded the largest loss of $6,626.39, while Sean Miller generated the highest sales among the loss-making customers at $25,043.05. These findings suggest that high sales do not always result in high profitability. The business should review discounts, product mix, and order-level costs for customers with repeated losses before making decisions about pricing or customer retention.

In [ ]:
#Compare profitable and unprofitable customers
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

bottom_profit_customers["Total_Profit"].sort_values().plot(
    kind="bar"
)

plt.title("10 Least Profitable Customers")
plt.xlabel("Customer")
plt.ylabel("Total Profit ($)")
plt.xticks(rotation=45, ha="right")
plt.axhline(y=0, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

top_profit_customers["Total_Profit"].sort_values().plot(
    kind="bar"
)

plt.title("10 Most Profitable Customers")
plt.xlabel("Customer")
plt.ylabel("Total Profit ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#let's perform data quality checks.
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())

# Check date range
print("\nOrder date range:")
print(df["Order Date"].min())
print(df["Order Date"].max())

In [ ]:
#Check future orders
# Check orders after November 9, 2026

future_orders = df[
    df["Order Date"] > "2026-11-09"
]

print("Future order records:", len(future_orders))

display(
    future_orders[
        ["Order ID", "Order Date", "Sales", "Profit"]
    ].head(10)
)

In [ ]:
# Count all future-dated records
print("Future records:", len(future_orders))

# Count unique future orders
print("Unique future orders:",
      future_orders["Order ID"].nunique())

# Check date range
print("First future date:",
      future_orders["Order Date"].min())

print("Last future date:",
      future_orders["Order Date"].max())

# Total future sales and profit
print("Future sales:",
      round(future_orders["Sales"].sum(), 2))

print("Future profit:",
      round(future_orders["Profit"].sum(), 2))

In [ ]:
# Keep records up to November 9, 2026
analysis_df = df[
    df["Order Date"] <= "2026-11-09"
].copy()

print("Original records:", len(df))
print("Historical records:", len(analysis_df))

print("Historical unique orders:",
      analysis_df["Order ID"].nunique())

print("Historical sales:",
      round(analysis_df["Sales"].sum(), 2))

print("Historical profit:",
      round(analysis_df["Profit"].sum(), 2))

print("Historical date range:",
      analysis_df["Order Date"].min(),
      "to",
      analysis_df["Order Date"].max())

In [ ]:
#Recalculate annual performance
annual_analysis = analysis_df.groupby(
    analysis_df["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")

)

annual_analysis["Profit_Margin_%"] = (
    annual_analysis["Total_Profit"] /
    annual_analysis["Total_Sales"] * 100
)
annual_analysis["Sales_Growth_%"] = (
    annual_analysis["Total_Sales"].pct_change() * 100
)
annual_analysis.round(2)


The annual performance analysis shows that sales declined slightly in 2024 but recovered strongly in 2025, increasing by approximately 29.8%. Profitability also improved, with the profit margin rising from 10.46% in 2023 to 13.46% in 2025. For 2026, the business generated $580,551.63 in sales through November 9. However, this figure represents a partial year and should not be directly compared with previous full-year results.

In [ ]:
#Compare 2025 and 2026 year-to-date
# Filter January 1 to November 9 for both years 09/11/2026
ytd_comparison = analysis_df[
    (analysis_df["Order Date"].dt.month <= 11) &
    (
        (analysis_df["Order Date"].dt.year == 2025) |
        (
            (analysis_df["Order Date"].dt.year == 2026) &
            (analysis_df["Order Date"].dt.day <= 9)
        )
    )
].copy()

# Aggregate by year
ytd_analysis = ytd_comparison.groupby(
    ytd_comparison["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

ytd_analysis["Profit_Margin_%"] = (
    ytd_analysis["Total_Profit"] /
    ytd_analysis["Total_Sales"] * 100
)

ytd_analysis.round(2)

In [ ]:
ytd_comparison = analysis_df[
    (
        (analysis_df["Order Date"] >= "2025-01-01") &
        (analysis_df["Order Date"] <= "2025-11-09")
    ) |
    (
        (analysis_df["Order Date"] >= "2026-01-01") &
        (analysis_df["Order Date"] <= "2026-11-09")
    )
].copy()

ytd_analysis = ytd_comparison.groupby(
    ytd_comparison["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

ytd_analysis["Profit_Margin_%"] = (
    ytd_analysis["Total_Profit"] /
    ytd_analysis["Total_Sales"] * 100
)

ytd_analysis["Sales_Growth_%"] = (
    ytd_analysis["Total_Sales"].pct_change() * 100
)

ytd_analysis.round(2)

In [ ]:
import matplotlib.pyplot as plt

ytd_analysis[["Total_Sales", "Total_Profit"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Year-to-Date Sales and Profit: 2025 vs 2026")
plt.xlabel("Year")
plt.ylabel("Amount ($)")
plt.xticks(rotation=0)
plt.legend(["Total Sales", "Total Profit"])
plt.tight_layout()
plt.show()

In [ ]:
ytd_analysis["Profit_Margin_%"].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Profit Margin Comparison: 2025 vs 2026")
plt.xlabel("Year")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Discount Analysis
discount_analysis = analysis_df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),   
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean"),  
    Total_Orders=("Order ID", "nunique"),
    Loss_Orders=("Profit", lambda x: (x < 0).sum())
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"] /
    discount_analysis["Total_Sales"]* 100 

)
discount_analysis.round(2)

##Professional management recommendation
Recommendation: Review discount policies, particularly discounts of 30% or higher, which are associated with negative aggregate profitability in this dataset. Prioritize profitable discount levels and evaluate high-discount transactions by product and customer segment before approving promotions.

In [ ]:
#Product Category Analysis

category_analysis = analysis_df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

category_analysis["Profit_Margin_%"] = (
    category_analysis["Total_Profit"] /
    category_analysis["Total_Sales"] * 100
)

category_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)


Preliminary business findings
Category	Sales	Profit margin
Technology	$374,608.88	18.64%
Office Supplies	$318,764.91	17.73%
Furniture	$352,457.40	1.82%

What the results suggest:
1. Technology has the highest profit margin at 18.64%.
2. Office Supplies has a strong profit margin of 17.73%.
3. Furniture has a very low profit margin of 1.82%, despite generating $352,457.40 in sales.
Furniture deserves further investigation.

In [ ]:

#Top 10 Loss-Making Products using analysis_df
product_analysis = analysis_df.groupby(
    ["Product ID", "Product Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Total_Orders=("Order ID", "nunique")
)

product_analysis["Profit_Margin_%"] = (
    product_analysis["Total_Profit"] /
    product_analysis["Total_Sales"] * 100
)

loss_products = product_analysis[
    product_analysis["Total_Profit"] < 0
].sort_values("Total_Profit")

loss_products.head(10).round(2)


##Management recommendation
Recommendation: Management should review the pricing, discount policies, and costs of the top loss-making products. Particular attention should be given to the Cubify CubeX printers, which generate substantial losses despite having recorded sales. Management should consider renegotiating supplier costs, adjusting selling prices, or discontinuing products if profitability cannot be improved.

In [ ]:
#Create a Top 10 Loss Products chart
import matplotlib.pyplot as plt
loss_products.head(10)["Total_Profit"].sort_values().plot(
    kind="barh",
    figsize=(10, 6)
)
plt.yticks(
    range(10),
    loss_products.head(10)["Total_Profit"].sort_values().index.get_level_values("Product ID")
)
plt.title("Top 10 Loss-Making Products")
plt.xlabel("Total Profit ($)")
plt.ylabel("Product ID")
plt.tight_layout()
plt.show()


In [ ]:
#Discount analysis by loss-making product
loss_product_discount = analysis_df[
    analysis_df["Product ID"].isin(
        loss_products.head(10).index.get_level_values("Product ID")
    )
].groupby(["Product ID","Product Name", "Discount"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)

loss_product_discount.round(2)

##Recommendation: 
Management should review products receiving discounts of 40% or higher, particularly the Cubify CubeX printers and selected office equipment. The company should evaluate product costs, selling prices, and discount approval policies. Lower-discount transactions show that some of these products can be profitable, so management should investigate whether revised pricing can improve profitability before discontinuing them.

In [ ]:
#Customer Segment Analysis
segment_analysis = analysis_df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

segment_analysis["Profit_Margin_%"] = (
    segment_analysis["Total_Profit"] /
    segment_analysis["Total_Sales"] * 100
)

segment_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)

Recommendation: The company should maintain its Consumer customer base because it generates the highest total revenue and profit. However, management should investigate ways to improve its profit margin. The Home Office segment offers an opportunity for growth because it has the highest profit margin. Targeted marketing and customer retention strategies may help increase sales in this segment.

In [ ]:

segment_analysis["Profit_Margin_%"].sort_values().plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Profit Margin by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#Regional Analysis
region_analysis = analysis_df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

region_analysis["Profit_Margin_%"] = (
    region_analysis["Total_Profit"] /
    region_analysis["Total_Sales"] * 100
)

region_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)

What does this tell management?
1. East: Highest total sales, total profit, and profit margin.
2. West: Strong performance, with the second-highest profit.
3. Central: Moderate performance, with room to improve profitability.
4. South: Lowest sales, profit, and profit margin. This region deserves further investigation.
The South's 7.95% margin is substantially below the East's 14.41%.
##Recommendation: 
The company should maintain its strong performance in the East and West regions. Management should investigate the South region's lower profit margin by reviewing product mix, discount policies, and operating costs. Improving profitability in the South may help increase overall business performance.

In [ ]:

region_analysis["Profit_Margin_%"].sort_values().plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Profit Margin by Region")
plt.xlabel("Region")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#Monthly Sales Trends
analysis_df["Order Date"] = pd.to_datetime(analysis_df["Order Date"])
monthly_analysis = analysis_df.groupby(
    analysis_df["Order Date"].dt.to_period("M")
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

monthly_analysis.tail(12).round(2)

Thanks! Your monthly analysis has produced the results. However, I noticed one important issue we should investigate before creating the chart.

Your table includes December 2026, but we previously defined analysis_df to exclude orders after November 9, 2026.

That means December 2026 should not appear.


In [ ]:
print("Latest Order Date:", analysis_df["Order Date"].max())

print(
    "Orders after Nov 9, 2026:",
    (analysis_df["Order Date"] > "2026-11-09").sum()
)

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
analysis_df = df[
    df["Order Date"] <= pd.Timestamp("2026-11-09")
].copy()

print("Total records:", len(analysis_df))
print("Latest Order Date:", analysis_df["Order Date"].max())
print(
    "Future orders:",
    (analysis_df["Order Date"] > "2026-11-09").sum()
)

In [ ]:
monthly_analysis = analysis_df.groupby(
    analysis_df["Order Date"].dt.to_period("M")
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

monthly_analysis.tail(12).round(2)

November only includes data through November 9, so it is not a full-month result.

Initial business observations:
- September had the highest sales in the displayed 12-month period.
- October sales were slightly lower than September.
- April had particularly low profit ($957.53), despite sales of $36,779.04.
- November's figures are incomplete and should not be compared directly with full months.

In [ ]:
monthly_analysis["Total_Sales"].plot(
    kind="line",
    figsize=(12, 5),
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
monthly_analysis["Total_Profit"].plot(
    kind="line",
    figsize=(12, 5),
    marker="o"
)

plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Total Profit ($)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
monthly_analysis["Profit_Margin_%"] = (
    monthly_analysis["Total_Profit"] /
    monthly_analysis["Total_Sales"] * 100
)

monthly_analysis.tail(12).round(2)

Monthly Sales and Profitability Analysis

The monthly analysis shows fluctuations in both sales and profitability during the reviewed period.

March 2026 achieved the highest profit margin at 24.72%, while April 2026 recorded a significantly lower margin of 2.60%, despite generating $36,779.04 in sales.

September 2026 generated the highest monthly sales of $88,064.53. However, its profit margin was 12.55%.

Recommendation: Management should investigate the reasons for the sharp decline in April's profitability, review discount policies, and monitor product-level profit margins to improve overall business performance.

Note: November 2026 data covers only November 1–9 and should not be compared directly with complete months.

In [ ]:
#Shipping analysis
analysis_df["Ship Date"] = pd.to_datetime(analysis_df["Ship Date"])
analysis_df["Order Date"] = pd.to_datetime(analysis_df["Order Date"])

analysis_df["Delivery_Days"] = (
    analysis_df["Ship Date"] - analysis_df["Order Date"]
).dt.days
shipping_analysis = analysis_df.groupby("Ship Mode").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Average_Delivery_Days=("Delivery_Days", "mean")
)

shipping_analysis["Profit_Margin_%"] = (
    shipping_analysis["Total_Profit"] /
    shipping_analysis["Total_Sales"] * 100
)

shipping_analysis.round(2)

Finding 1 — First Class has the highest profit margin.

First Class achieves a 14.85% margin, the highest among the four shipping modes.

Finding 2 — Standard Class is the most commonly used.

It accounts for 2,818 orders, but has the lowest margin at 12.18%.

Finding 3 — Same Day is the fastest shipping mode.

Its average order-to-ship interval is 0.04 days.

Important: These figures show an association between shipping mode and profitability. They do not prove that shipping mode causes higher or lower profit. We also do not have shipping-cost data to determine which option is most cost-effective.

In [ ]:
shipping_analysis["Profit_Margin_%"].plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Profit Margin by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Shipping Mode Performance Analysis

First Class recorded the highest profit margin at 14.85%, while Standard Class generated the highest order volume, with 2,818 orders.

Standard Class had the lowest profit margin at 12.18%. Same Day shipping had the shortest average order-to-ship interval.

**Recommendation:** Management should review the profitability and costs associated with each shipping mode before making changes to shipping policies.

In [ ]:
##Calculate customer performance
customer_analysis = analysis_df.groupby(
    ["Customer ID", "Customer Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Total_Quantity=("Quantity", "sum")
)

customer_analysis["Profit_Margin_%"] = (
    customer_analysis["Total_Profit"] /
    customer_analysis["Total_Sales"] * 100
)

top_customers = customer_analysis.sort_values(
    "Total_Sales",
    ascending=False
).head(10)

top_customers.round(2)

Important business observations:
1. Sean Miller generates the highest sales, but the customer relationship is currently unprofitable.
2. Tamara Chand generates less revenue but much higher profit.
3. Raymond Buch also performs strongly, with a 46.15% profit margin.
4. Ken Lonsdale has a relatively low profit margin of 5.69%, despite placing 11 orders.
This demonstrates an important business lesson:

High sales do not always mean high profitability.

In [ ]:
top_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=False
).head(10)

top_profit_customers.round(2)

In [ ]:
bottom_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=True
).head(10)

bottom_profit_customers.round(2)

In [ ]:
bottom_profit_customers["Total_Profit"].sort_values().plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Top 10 Loss-Making Customers")
plt.xlabel("Total Profit ($)")
plt.ylabel("Customer Name")

plt.yticks(
    range(10),
    bottom_profit_customers["Total_Profit"]
    .sort_values()
    .index.get_level_values("Customer Name")
)

plt.tight_layout()
plt.show()

In [ ]:
##Final project data validation
total_sales = analysis_df["Sales"].sum()
total_profit = analysis_df["Profit"].sum()
total_orders = analysis_df["Order ID"].nunique()

profit_margin = total_profit / total_sales * 100

print("Total Records:", len(analysis_df))
print("Unique Orders:", total_orders)
print("Total Sales: $", round(total_sales, 2))
print("Total Profit: $", round(total_profit, 2))
print("Profit Margin:", round(profit_margin, 2), "%")

Next stage: Professional portfolio presentation
We now need to turn your analysis into something an employer or Upwork client can understand.

I recommend three remaining tasks:
1. Management report: Executive summary, key findings, charts, and recommendations.
2. Notebook cleanup: Organize your Python code, headings, explanations, and charts.
3. Power BI dashboard: Present the most important KPIs and business insights visually.
My recommendation is to finish the written management report first, then build the Power BI dashboard using the same validated dataset.

##1. Management Report
# Retail Sales Analysis
**Python Data Analytics Portfolio Project**

## 1. Executive Summary

This project analyzes retail sales data to evaluate business performance, customer profitability, product performance, regional results, discount strategies, and shipping methods.

The analysis uses 9,390 records and 4,688 unique orders, covering January 3, 2023, through November 9, 2026.

### Key Performance Indicators

| Business KPI | Result |
|---|---:|
| Total Sales | $2,161,518.45 |
| Total Profit | $274,497.35 |
| Unique Orders | 4,688 |
| Total Records | 9,390 |
| Overall Profit Margin | 12.70% |

## 2. Key Business Findings

### Sales and Profitability
Sales increased by 27.46% in the first period of January 1–November 9, 2026, compared with the same period in 2025.

Profit increased by 24.58% over the same period. However, profit margin decreased slightly from 13.77% to 13.46%.

### Discount Performance
Discount levels of 30% and above recorded negative aggregate profit in this dataset. Management should review high-discount transactions, product costs, and discount policies.

### Customer Performance
Tamara Chand was the most profitable customer among the top 10 customers by sales, generating $8,981.32 in profit.

Sean Miller generated the highest sales among those top 10 customers ($25,043.05), but recorded a loss of $1,980.74.

### Regional Performance
The East region recorded the highest profit margin at 14.41%, while the South region had the lowest at 7.95%.

### Shipping Performance
First Class had the highest profit margin at 14.85%. Standard Class generated the largest order volume, with 2,818 orders.

### Monthly Performance
March 2026 achieved the highest monthly profit margin in the displayed 12-month period at 24.72%. April 2026 recorded a much lower margin of 2.60%, requiring further investigation.

## 3. Management Recommendations

1. Review high-discount transactions, particularly discounts of 30% or more.
2. Investigate products and customers generating negative profit.
3. Develop strategies to improve profitability in the South region.
4. Maintain relationships with high-profit customers.
5. Investigate the reasons for low monthly profit margins, especially in April 2026.
6. Monitor profit margins alongside sales growth when evaluating business performance.

## 4. Data Limitations

The original dataset contained 804 records dated after November 9, 2026. These records were excluded from the analysis because they were later than the selected reporting cutoff.

The dataset does not include shipping costs. Therefore, shipping cost efficiency cannot be evaluated.

The analysis identifies relationships and patterns in the data but does not establish causation.

In [ ]:
print("Overall sales:", analysis_df["Sales"].sum())

print(
    "Category sales:",
    category_analysis["Total_Sales"].sum()
)

print("Category record counts:")
print(analysis_df["Category"].value_counts())

In [ ]:
#Rebuild Category Analysis
category_analysis = analysis_df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Profit_Margin_=("Profit", "sum")
)

category_analysis["Profit_Margin_%"] = (
    category_analysis["Total_Profit"] /
    category_analysis["Total_Sales"] * 100
)

category_analysis = category_analysis.drop(
    columns=["Profit_Margin_"]
)

category_analysis.round(2)


In [ ]:
## Check the totals
print(
    "Overall Sales:",
    round(analysis_df["Sales"].sum(), 2)
)

print(
    "Category Sales:",
    round(category_analysis["Total_Sales"].sum(), 2)
)

In [ ]:
subcategory_analysis = analysis_df.groupby("Sub-Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

subcategory_analysis["Profit_Margin_%"] = (
    subcategory_analysis["Total_Profit"] /
    subcategory_analysis["Total_Sales"] * 100
)

subcategory_analysis = subcategory_analysis.sort_values(
    "Profit_Margin_%", ascending=False
).round(2)

display(subcategory_analysis)

print("Total Sales:", subcategory_analysis["Total_Sales"].sum())
print("Total Profit:", subcategory_analysis["Total_Profit"].sum())

In [ ]:
discount_analysis = analysis_df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Records=("Discount", "size"),
    Orders=("Order ID", "nunique")
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"]
    / discount_analysis["Total_Sales"] * 100
)

display(discount_analysis.round(2))

In [ ]:
high_discount_analysis = (
    analysis_df[analysis_df["Discount"] >= 0.30]
    .groupby(["Discount", "Sub-Category"])
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Records=("Discount", "size")
    )
)

display(high_discount_analysis.sort_values("Total_Profit").round(2))

In [ ]:
high_discount_df = analysis_df[analysis_df["Discount"] >= 0.30]

segment_analysis = high_discount_df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Records=("Discount", "size")
)

segment_analysis["Profit_Margin_%"] = (
    segment_analysis["Total_Profit"]
    / segment_analysis["Total_Sales"] * 100
)

region_analysis = high_discount_df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Records=("Discount", "size")
)

region_analysis["Profit_Margin_%"] = (
    region_analysis["Total_Profit"]
    / region_analysis["Total_Sales"] * 100
)

print("By customer segment:")
display(segment_analysis.round(2))

print("By region:")
display(region_analysis.round(2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Profit by category
category_profit = category_analysis["Total_Profit"]
category_colors = [
    "seagreen" if value >= 0 else "indianred"
    for value in category_profit
]

category_profit.plot(
    kind="bar",
    ax=axes[0],
    color=category_colors,
    title="Total Profit by Category"
)
axes[0].set_ylabel("Profit ($)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].axhline(0, color="black", linewidth=0.8)

# Profit by discount level
discount_profit = discount_analysis["Total_Profit"]
discount_colors = [
    "seagreen" if value >= 0 else "indianred"
    for value in discount_profit
]

discount_profit.plot(
    kind="bar",
    ax=axes[1],
    color=discount_colors,
    title="Total Profit by Discount Level"
)
axes[1].set_ylabel("Profit ($)")
axes[1].set_xlabel("Discount")
axes[1].tick_params(axis="x", rotation=45)
axes[1].axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("portfolio_charts.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
##Power BI
## Export the cleaned and analyzed data to CSV for Power BI visualization
analysis_df.to_csv("../data/cleaned_retail_sales_data.csv", index=False)